In [0]:
from pyspark.sql import functions as F

BASE = "workspace.retail"

# Column types for each table.
SCHEMAS = {
    "categories": {
        "category_id": "int",
        "category_name": "string",
        "parent_category_id": "int",
    },
    "products": {
        "product_id": "int",
        "product_name": "string",
        "product_price": "decimal(10,2)",
        "category_id": "int",
    },
    "stores": {
        "store_id": "int",
        "store_city": "string",
        "store_state": "string",
    },
    "customers": {
        "customer_id": "int",
        "customer_name": "string",
        "customer_city": "string",
        "customer_state": "string",
        "segment": "string",
        "created_at": "timestamp",
    },
    "orders": {
        "order_id": "int",
        "customer_id": "int",
        "store_id": "int",
        "status": "string",
        "created_at": "timestamp",
        "updated_at": "timestamp",
    },
    "order_items": {
        "order_item_id": "int",
        "order_id": "int",
        "product_id": "int",
        "quantity": "int",
        "unit_price": "decimal(10,2)",
    },
}

KEYS = {
    "categories": "category_id",
    "products": "product_id",
    "stores": "store_id",
    "customers": "customer_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

# Explicit rules for this project's analytical data.
REQUIRED = {
    "categories": ["category_id", "category_name"],
    "products": [
        "product_id", "product_name", "product_price", "category_id"
    ],
    "stores": ["store_id", "store_city", "store_state"],
    "customers": [
        "customer_id", "customer_name", "customer_city",
        "customer_state", "segment", "created_at"
    ],
    "orders": [
        "order_id", "customer_id", "store_id",
        "status", "created_at", "updated_at"
    ],
    "order_items": [
        "order_item_id", "order_id", "product_id",
        "quantity", "unit_price"
    ],
}


def add_error(df, condition, message):
    """Append a reason when a row fails a check."""
    return df.withColumn(
        "_errors",
        F.concat(
            F.col("_errors"),
            F.when(
                F.coalesce(condition, F.lit(False)),
                F.array(F.lit(message))
            ).otherwise(F.expr("CAST(array() AS ARRAY<STRING>)"))
        )
    )


def prepare(table):
    raw = spark.table(f"{BASE}.bronze_{table}")

    # Keep the original record for investigating rejected rows.
    df = raw.withColumn(
        "_raw_record",
        F.to_json(F.struct(*[F.col(c) for c in raw.columns]))
    ).withColumn(
        "_errors",
        F.expr("CAST(array() AS ARRAY<STRING>)")
    )

    for column, dtype in SCHEMAS[table].items():
        if dtype == "string":
            df = df.withColumn(column, F.trim(F.col(column)))
        else:
            # Invalid conversions become null instead of crashing.
            converted = F.expr(f"try_cast(`{column}` AS {dtype})")

            bad_conversion = (
                F.col(column).isNotNull()
                & (F.trim(F.col(column)) != "")
                & converted.isNull()
            )

            df = add_error(
                df, bad_conversion, f"Invalid type: {column}"
            )
            df = df.withColumn(column, converted)

    for column in REQUIRED[table]:
        missing = F.col(column).isNull()

        if SCHEMAS[table][column] == "string":
            missing = missing | (F.col(column) == "")

        df = add_error(df, missing, f"Missing value: {column}")

    # Reject every row sharing a duplicate key.
    # We do not arbitrarily choose which version is correct.
    key = KEYS[table]

    duplicate_keys = (
        df.filter(F.col(key).isNotNull())
        .groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .select(key)
        .withColumn("_duplicate", F.lit(True))
    )

    df = df.join(duplicate_keys, on=key, how="left")
    df = add_error(df, F.col("_duplicate"), f"Duplicate key: {key}")

    return df.drop("_duplicate")


def check_reference(df, column, parent_df, parent_key):
    """Check a foreign key against accepted parent records."""
    parent_ids = (
        parent_df.select(F.col(parent_key).alias(column))
        .filter(F.col(column).isNotNull())
        .distinct()
        .withColumn("_parent_exists", F.lit(True))
    )

    df = df.join(parent_ids, on=column, how="left")

    # Nullability is checked separately by REQUIRED.
    df = add_error(
        df,
        F.col(column).isNotNull() & F.col("_parent_exists").isNull(),
        f"Unknown reference: {column}"
    )

    return df.drop("_parent_exists")


def save_results(table, df):
    valid = (
        df.filter(F.size("_errors") == 0)
        .select(*SCHEMAS[table].keys())
    )

    rejected = (
        df.filter(F.size("_errors") > 0)
        .withColumn("rejection_reason", F.concat_ws("; ", "_errors"))
        .drop("_errors")
    )

    for name, result in [
        (f"silver_{table}", valid),
        (f"rejected_{table}", rejected),
    ]:
        (
            result.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{BASE}.{name}")
        )

    source_count = spark.table(f"{BASE}.bronze_{table}").count()
    valid_count = spark.table(f"{BASE}.silver_{table}").count()
    rejected_count = spark.table(f"{BASE}.rejected_{table}").count()

    assert source_count == valid_count + rejected_count, (
        f"Row-count mismatch for {table}"
    )

    print(
        f"{table}: {source_count} source | "
        f"{valid_count} valid | {rejected_count} rejected"
    )

    return spark.table(f"{BASE}.silver_{table}")


# 1. Categories
categories = prepare("categories")

categories = add_error(
    categories,
    F.col("category_id") == F.col("parent_category_id"),
    "Category cannot be its own parent"
)

# Remove references to rejected/missing parents.
# Repeat so rejection propagates through the category hierarchy.
while True:
    previous_errors = categories.filter(
        F.size("_errors") > 0
    ).count()

    accepted_parents = categories.filter(F.size("_errors") == 0)

    categories = check_reference(
        categories, "parent_category_id",
        accepted_parents, "category_id"
    )

    current_errors = categories.filter(
        F.size("_errors") > 0
    ).count()

    if current_errors == previous_errors:
        break

categories = save_results("categories", categories)


# 2. Stores
stores = save_results("stores", prepare("stores"))


# 3. Customers
customers = prepare("customers")

customers = add_error(
    customers,
    ~F.col("segment").isin("Regular", "Premium"),
    "Unsupported customer segment"
)

customers = save_results("customers", customers)


# 4. Products
products = prepare("products")

products = add_error(
    products,
    F.col("product_price") < 0,
    "Price cannot be negative"
)

products = check_reference(
    products, "category_id", categories, "category_id"
)

products = save_results("products", products)


# 5. Orders
orders = prepare("orders")
orders = orders.withColumn("status", F.upper(F.col("status")))

orders = add_error(
    orders,
    ~F.col("status").isin(
        "PENDING", "PAID", "SHIPPED", "CANCELLED", "RETURNED"
    ),
    "Unsupported order status"
)

orders = add_error(
    orders,
    F.col("updated_at") < F.col("created_at"),
    "updated_at is earlier than created_at"
)

orders = check_reference(
    orders, "customer_id", customers, "customer_id"
)

orders = check_reference(
    orders, "store_id", stores, "store_id"
)

orders = save_results("orders", orders)


# 6. Order items
items = prepare("order_items")

items = add_error(
    items,
    F.col("quantity") <= 0,
    "Quantity must be greater than zero"
)

items = add_error(
    items,
    F.col("unit_price") < 0,
    "Unit price cannot be negative"
)

items = check_reference(
    items, "order_id", orders, "order_id"
)

items = check_reference(
    items, "product_id", products, "product_id"
)

items = save_results("order_items", items)

print("Silver processing complete.")

categories: 2 source | 2 valid | 0 rejected
stores: 3 source | 3 valid | 0 rejected
customers: 50 source | 50 valid | 0 rejected
products: 5 source | 5 valid | 0 rejected
orders: 2001 source | 2001 valid | 0 rejected
order_items: 3055 source | 3055 valid | 0 rejected
Silver processing complete.


In [0]:
display(spark.table("workspace.retail.rejected_orders"))

store_id,customer_id,order_id,status,created_at,updated_at,_raw_record,rejection_reason
